# 02 — Sanity Check: Logistic Regression on Raw EEG

Quick upstream check on a small slice of data: is the label/preprocessing pipeline sound before moving to the transformer smoke test? **Not a reported baseline** — a diagnostic only.


## Imports

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange
import mne
from mne.io import read_raw_edf, concatenate_raws


## Config and subject-level split

Small subject count on purpose — this is a fast check, not the real evaluation. Split happens on the **subject list**, before any data is loaded, so train/test subjects never overlap.

In [ ]:
N_SUBJECTS = 20
RUNS = [4, 8, 12]        # imagined left/right fist
TMIN, TMAX = 0.0, 4.0
TEST_FRACTION = 0.25

subjects = [i for i in range(1, 110) if i not in [88, 89, 92, 100]][:N_SUBJECTS]
n_test = max(1, int(TEST_FRACTION * len(subjects)))
train_subjects = subjects[:-n_test]
test_subjects = subjects[-n_test:]

print("train subjects:", train_subjects)
print("test subjects:", test_subjects)


## Load epochs directly from MNE, per subject group

Same loading logic as the standalone sanity-check script: sfreq assertion, event mapping built from `event_id_dict` (not hardcoded, to avoid the boundary-annotation code-shift issue), rest excluded so this is a clean binary left/right task.

In [ ]:
def load_epochs(subject_list):
    X_list, y_list = [], []
    for subject in subject_list:
        file_paths = mne.datasets.eegbci.load_data(subject, RUNS)
        raw_objects = [read_raw_edf(fp, preload=True) for fp in file_paths]
        raw = concatenate_raws(raw_objects)

        assert raw.info['sfreq'] == 160, f"subject {subject} has unexpected sfreq"

        raw.filter(l_freq=8, h_freq=30)

        events, event_id_dict = mne.events_from_annotations(raw)
        mapping = {
            'left_fist': event_id_dict['T1'],
            'right_fist': event_id_dict['T2'],
        }

        epochs = mne.Epochs(raw, events, event_id=mapping,
                             tmin=TMIN, tmax=TMAX, baseline=None, preload=True)

        X = epochs.get_data().astype(np.float32)
        y = epochs.events[:, -1]

        X_list.append(X)
        y_list.append(y)

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_all, y_all


print("Loading train subjects...")
X_train, y_train = load_epochs(train_subjects)
print("Loading test subjects...")
X_test, y_test = load_epochs(test_subjects)

# volts -> microvolts, numerical conditioning
X_train = X_train * 1e6
X_test = X_test * 1e6

print("X_train:", X_train.shape, "  X_test:", X_test.shape)


## Flatten for a linear model

`(n_epochs, n_channels, n_times)` → `(n_epochs, n_channels * n_times)`, since `LogisticRegression` needs 2D input.

In [ ]:
X_train_flat = rearrange(X_train, 'batch channel time -> batch (channel time)')
X_test_flat = rearrange(X_test, 'batch channel time -> batch (channel time)')
print("train reshaped:", X_train_flat.shape)
print("test reshaped:", X_test_flat.shape)


## Diagnostics — confirm the data isn't degenerate before fitting anything

Checks: no NaNs, real per-feature variance (not constant/dead features), and balanced classes in both splits.

In [ ]:
print("any NaNs (train):", np.isnan(X_train_flat).any())
print("any NaNs (test):", np.isnan(X_test_flat).any())
print("feature std range (train):", X_train_flat.std(axis=0).min(), "-", X_train_flat.std(axis=0).max())
print()
print("train label counts:", np.unique(y_train, return_counts=True))
print("test label counts:", np.unique(y_test, return_counts=True))


## Fit across a range of regularization strengths

With tens of thousands of raw features and a small number of samples, an unregularized linear model will trivially hit ~1.0 train accuracy regardless of whether the signal is real (curse of dimensionality) — the informative number is **test accuracy**, and how it moves as regularization (`C`) tightens.

In [ ]:
C_values = [1.0, 0.1, 0.01, 0.001]
train_accs, test_accs = [], []

for C in C_values:
    model = LogisticRegression(max_iter=1000, C=C)
    model.fit(X_train_flat, y_train)
    train_acc = model.score(X_train_flat, y_train)
    test_acc = model.score(X_test_flat, y_test)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    print(f"C={C:<6} train_acc={train_acc:.3f}  test_acc={test_acc:.3f}  (chance=0.500)")


### Plot: accuracy vs. regularization strength

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(C_values, train_accs, marker='o', label='train accuracy')
ax.plot(C_values, test_accs, marker='o', label='test accuracy')
ax.axhline(0.5, color='gray', linestyle='--', label='chance level')
ax.set_xscale('log')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('accuracy')
ax.set_title('Logistic regression: train vs. test accuracy across C\n(subject-level split)')
ax.legend()
plt.tight_layout()
plt.show()


### Confusion matrix (at the most regularized setting)

Checked explicitly rather than trusting accuracy alone — this rules out the failure mode where a model just predicts one class for everything and still scores near 50% by accident on balanced binary data.

In [ ]:
best_C = C_values[-1]
model = LogisticRegression(max_iter=1000, C=best_C)
model.fit(X_train_flat, y_train)
preds = model.predict(X_test_flat)
cm = confusion_matrix(y_test, preds)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['left_fist', 'right_fist'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['left_fist', 'right_fist'])
ax.set_xlabel('predicted'); ax.set_ylabel('actual')
ax.set_title(f'Confusion matrix (C={best_C}, subject-level split)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.colorbar(im)
plt.tight_layout()
plt.show()


## Conclusion

- Train/test split is now genuinely **subject-level** (subjects split before loading, no epoch from a test subject ever seen during training) — the result here is trustworthy in a way the earlier index-sliced version was not.
- No NaNs, real feature variance, balanced classes in both splits — pipeline is not obviously broken.
- Train accuracy pinned near 1.0 regardless of `C` is expected given the feature/sample ratio here, not evidence of a working classifier on its own.
- Test accuracy trend and confusion-matrix balance are what actually matter — look for a non-degenerate matrix and any consistent above-chance trend.